# 04 - Heston Calibration

**Objective:** Fit Heston parameters to a selected market implied-volatility smile.

**Inputs:** `data/processed/AAPL_surface_checked.csv` and `data/processed/AAPL_iv_fit.csv` from notebook 01, with fallback to the frozen sample under `data/sample/`.

**Outputs:** Fitted parameters, model-vs-market IV comparison, and residual table.

**Key takeaway:** The objective minimizes error in implied-volatility space so calibration quality can be interpreted directly in volatility points.




In [ ]:
from pathlib import Path
import os
import sys
import tempfile

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "heston_matplotlib_cache"))

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd

from src.calibration import calibrate_heston_to_smile, select_single_expiry_target
from src.heston import HestonParams, get_torch_device
from src.plotting import fit_iv_surface, plot_heston_fit

In [ ]:
TICKER = "AAPL"
checked_path = PROJECT_ROOT / "data" / "processed" / f"{TICKER}_surface_checked.csv"
iv_fit_path = PROJECT_ROOT / "data" / "processed" / f"{TICKER}_iv_fit.csv"
sample_checked_path = PROJECT_ROOT / "data" / "sample" / f"{TICKER}_canonical_surface_checked.csv"
sample_iv_fit_path = PROJECT_ROOT / "data" / "sample" / f"{TICKER}_canonical_iv_fit.csv"

if checked_path.exists():
    df_path = checked_path
elif sample_checked_path.exists():
    df_path = sample_checked_path
else:
    raise FileNotFoundError("Run notebook 01 first or keep the canonical sample CSV under data/sample/.")

df = pd.read_csv(df_path, parse_dates=["expiry"])
spot = float((df["strike"] / df["moneyness"]).median())
rate = float(df["r"].dropna().median()) if "r" in df else 0.0
dividend = float(df["q"].dropna().median()) if "q" in df else 0.0

if iv_fit_path.exists():
    iv_fit_df = pd.read_csv(iv_fit_path, parse_dates=["expiry"])
elif sample_iv_fit_path.exists():
    iv_fit_df = pd.read_csv(sample_iv_fit_path, parse_dates=["expiry"])
else:
    iv_fit_df = fit_iv_surface(df, outdir=FIG_DIR)

print(f"Loaded {df_path.relative_to(PROJECT_ROOT)}")
print(f"spot={spot:.4f}, r={rate:.4f}, q={dividend:.4f}")



## Select Calibration Target

The target smile is chosen from the 3 to 9 month region where option quotes are usually less noisy than very short-dated expiries.

In [ ]:
target_expiry, maturity, all_strikes, all_target_iv = select_single_expiry_target(iv_fit_df, spot=spot)
all_moneyness = all_strikes / spot
central_idx = np.where((all_moneyness >= 0.89) & (all_moneyness <= 1.15))[0]
if len(central_idx) > 15:
    central_idx = central_idx[np.linspace(0, len(central_idx) - 1, 15).round().astype(int)]

strikes = all_strikes[central_idx]
target_iv = all_target_iv[central_idx]

target_rows = df[df["expiry"] == target_expiry]
rate = float(target_rows["r"].dropna().median()) if "r" in target_rows else rate
dividend = float(target_rows["q"].dropna().median()) if "q" in target_rows else dividend

print(f"Target expiry: {target_expiry.date()}, T={maturity:.4f}, points={len(strikes)}")
print(f"Target moneyness: {strikes.min() / spot:.2f} to {strikes.max() / spot:.2f}")
print(f"Calibration r={rate:.4f}, q={dividend:.4f}")



## Fit Heston Parameters

The objective minimizes mean squared error in implied-volatility space. The default path count is deliberately moderate so the notebook is runnable. Increase `n_paths` and `maxiter` for a final research run.

In [ ]:
device = get_torch_device(prefer_gpu=True)
initial = HestonParams(v0=0.04, kappa=2.0, theta=0.04, sigma=0.30, rho=-0.70)

params, objective, success, message, model_iv = calibrate_heston_to_smile(
    spot=spot,
    rate=rate,
    dividend=dividend,
    maturity=maturity,
    strikes=strikes,
    target_iv=target_iv,
    initial=initial,
    n_steps=60,
    n_paths=6_000,
    maxiter=30,
    seed=42,
    device=device,
)

residuals = model_iv - target_iv
rmse = float(np.sqrt(np.mean(residuals**2)))
mae = float(np.mean(np.abs(residuals)))
max_abs_residual = float(np.max(np.abs(residuals)))

print(success, message)
print(f"objective={objective:.8f}")
print(f"IV RMSE={rmse:.4f}; MAE={mae:.4f}; max abs residual={max_abs_residual:.4f}")
print(params)



In [ ]:
plot_heston_fit(strikes, target_iv, model_iv, target_expiry, outdir=FIG_DIR)

calibration_table = pd.DataFrame({
    "strike": strikes,
    "moneyness": strikes / spot,
    "target_iv": target_iv,
    "heston_iv": model_iv,
    "iv_error": model_iv - target_iv,
})
calibration_table.to_csv(PROJECT_ROOT / "reports" / "calibration_residuals.csv", index=False)
calibration_table.head()



## Interpretation

A good calibration should match the overall smile level and skew, not every noisy point. The fitted correlation `rho` controls the leverage/skew effect, `sigma` controls volatility-of-volatility, `theta` is long-run variance, `kappa` controls mean reversion, and `v0` is the initial variance.

## Takeaway

The calibrated smile fit should be read with the residual table: the model captures the central smile level, while residuals identify where the one-expiry setup is too limited.

